#### Read the data

In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler

df = pd.read_csv("dataset/1_Daily_minimum_temps.csv")   # columns: Date, Temp

df["Temp"] = df["Temp"].astype(str).str.replace("?", "", regex=False)
df["Temp"] = df["Temp"].astype("float32")

temps = df["Temp"].values
temps

array([20.7, 17.9, 18.8, ..., 13.5, 15.7, 13. ],
      shape=(3650,), dtype=float32)

#### Normalize the data

In [3]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
temps_scaled = scaler.fit_transform(temps.reshape(-1, 1)).flatten()


#### Create sequences (12 days → next day)

In [4]:
import numpy as np

SEQ_LEN = 12
X_list, y_list = [], []

for i in range(len(temps_scaled) - SEQ_LEN):
    X_list.append(temps_scaled[i:i+SEQ_LEN])
    y_list.append(temps_scaled[i+SEQ_LEN])

X = np.array(X_list)
y = np.array(y_list)


#### Reshape X for RNN input

In [5]:
X = X.reshape(X.shape[0], X.shape[1], 1)


#### Split into train, validation, test (70% / 15% / 15%)

In [6]:
n = len(X)

train_end = int(n * 0.70)
val_end   = int(n * 0.85)

X_train = X[:train_end]
y_train = y[:train_end]

X_val = X[train_end:val_end]
y_val = y[train_end:val_end]

X_test = X[val_end:]
y_test = y[val_end:]


#### Build the Stacked RNN model

In [8]:
import tensorflow as tf

# First RNN layer:
# return_sequences=True → keep the full sequence so the next RNN layer can read it.
# (The second RNN needs a sequence, not a single vector.)
#
# Second RNN layer:
# return_sequences=False → collapse the sequence into one final vector.
# (The Dense layer expects one vector, not a sequence.)
#
# Dense layer:
# Produces the final prediction (next day's temperature).

model = tf.keras.Sequential([
    tf.keras.layers.SimpleRNN(64, return_sequences=True),
    tf.keras.layers.SimpleRNN(32, return_sequences=False),
    tf.keras.layers.Dense(1)
])


#### Compile the model

In [9]:
model.compile(optimizer="adam", loss="mse", metrics=["mae"])


#### Show model summary

In [10]:
model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ simple_rnn (SimpleRNN)               │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ simple_rnn_1 (SimpleRNN)             │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

#### Train the model

In [11]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=32
)


Epoch 1/50
80/80 ━━━━━━━━━━━━━━━━━━━━ 11s 24ms/step - loss: 0.0218 - mae: 0.1112 - val_loss: 0.0112 - val_mae: 0.0844
Epoch 2/50
80/80 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - loss: 0.0110 - mae: 0.0823 - val_loss: 0.0088 - val_mae: 0.0741
Epoch 3/50
80/80 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 0.0097 - mae: 0.0775 - val_loss: 0.0115 - val_mae: 0.0873
Epoch 4/50
80/80 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.0099 - mae: 0.0778 - val_loss: 0.0114 - val_mae: 0.0868
Epoch 5/50
80/80 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.0094 - mae: 0.0761 - val_loss: 0.0100 - val_mae: 0.0807
Epoch 6/50
80/80 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 0.0099 - mae: 0.0783 - val_loss: 0.0082 - val_mae: 0.0719
Epoch 7/50
80/80 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 0.0094 - mae: 0.0762 - val_loss: 0.0081 - val_mae: 0.0707
Epoch 8/50
80/80 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 0.0093 - mae: 0.0759 - val_loss: 0.0082 - val_mae: 0.0705
Epoch 9/50
80/80 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: 0.00

#### Evaluate on train and test

In [12]:
train_loss, train_mae = model.evaluate(X_train, y_train, verbose=0)
test_loss, test_mae = model.evaluate(X_test, y_test, verbose=0)

print("Train MAE:", train_mae)
print("Test MAE:", test_mae)


Train MAE: 0.07617071270942688
Test MAE: 0.06961910426616669
